# Notebook exploratorio de las bases de datos del INER:
* `INER_COVID19_CostoPacientes_Econo.csv`
* `INER_COVID19_Pacientes_DiagnosticoComorbilidad.csv` 
* `INER_COVID19_TrabajoSocial.csv` 

## Cargaremos cada archivo y realizaremos un análisis exploratorio de los datos, buscando entender su estructura, identificar posibles problemas de calidad de datos y encontrar patrones o insights relevantes para nuestro análisis posterior.

In [2]:
import pandas as pd
from pathlib import Path
ruta_base = Path().absolute().parent
ruta_datos = ruta_base / 'data'
archivo_1 = 'INER_COVID19_CostoPacientes_Econo.csv'
archivo_2 = 'INER_COVID19_Pacientes_DiagnosticoComorbilidad.csv'
archivo_3 = 'INER_COVID19_TrabajoSocial.csv'

## 1. ..._CostoPacientes_Econo.csv
**Información económica de los pacientes**

In [4]:
df_economico = pd.read_csv(ruta_datos / archivo_1)
df_economico.info(memory_usage=False)

<class 'pandas.DataFrame'>
RangeIndex: 4632 entries, 0 to 4631
Data columns (total 24 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   EXP                               4552 non-null   str    
 1   NOMBRE_DEL_PACIENTE               4632 non-null   str    
 2   SEXO                              4632 non-null   str    
 3   EDAD                              4625 non-null   float64
 4   GRUPO_EDAD                        4625 non-null   str    
 5   RESULTADO                         3409 non-null   str    
 6   ETIQUETAS_COVID                   1550 non-null   str    
 7   MOTIVO_DE_EGRESO                  4614 non-null   str    
 8   FECHA_INGRESO_INER                4632 non-null   str    
 9   FECHA_DE_ALTA_MEJORIA             4632 non-null   str    
 10  DIAS_ESTANCIA                     4632 non-null   int64  
 11  GASTO_TOTAL                       4632 non-null   float64
 12  GASTO_DIARIO     

## 2. ..._Pacientes_DiagnosticoComorbilidad.csv
**Diagnósticos y comorbilidades de los pacientes**

In [3]:
df_comorbilidad = pd.read_csv(ruta_datos / archivo_2)
df_comorbilidad.info(memory_usage=False)

<class 'pandas.DataFrame'>
RangeIndex: 4278 entries, 0 to 4277
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   expediente            4278 non-null   int64  
 1   nombre                4278 non-null   str    
 2   fechaing              4278 non-null   str    
 3   fechaegr              4278 non-null   str    
 4   diagnosticoprincipal  4278 non-null   str    
 5   cie101                4278 non-null   str    
 6   diagnostico2          2927 non-null   str    
 7   cie102                2915 non-null   str    
 8   diagnostico3          1563 non-null   str    
 9   cie103                1561 non-null   str    
 10  diagnostico4          461 non-null    str    
 11  cie104                461 non-null    str    
 12  dx2                   2915 non-null   str    
 13  dx3                   1561 non-null   str    
 14  dx4                   461 non-null    str    
 15  obesidad              4278 non-n

Este CSV contiene información sobre las comorbilidades de los pacientes, sin embargo, sus columnas no están claramente definidas. Para entender mejor su contenido, es necesario explorar las columnas, sus valores únicos, valores nulos y su relación con otras variables.

De manera específica, las siguientes columnas están relacionadas con los diferentes diagnósticos que cada paciente recibió, segun el siguiente orden:
- `diagnosticoprincipal`, `cie101`. Principal diagnóstico (o número 1) del paciente.
- `diagnostico2`, `dx2`, `cie102`. Segundo diagnóstico del paciente.
- `diagnostico3`, `dx3`, `cie103`. Tercer diagnóstico del paciente.
- `diagnostico4`, `dx4`, `cie104`. Cuarto diagnóstico del paciente.

Exploramos el contenido de estas columnas en conjunto para asegurar que entre cada grupo reflejen el mismo diagnóstico, y para entender la cantidad de diagnósticos únicos que existen en cada una de ellas.

### 2.1 Columnas relacionadas con el **diagnóstico principal** del paciente:

In [22]:
# 1. Limpieza básica para el análisis, quitando espacios y unificando mayúsculas para el campo de diagnóstico principal
df_comorbilidad['d1_limpio'] = df_comorbilidad['diagnosticoprincipal'].str.upper().str.strip()

# 2. Resumen de Cardinalidad (Original vs Limpio)
resumen_cardinalidad = pd.DataFrame({
    'Campo': ['diagnosticoprincipal (Original)', 'diagnosticoprincipal (Limpio)', 'cie101'],
    'Valores Únicos': [
        df_comorbilidad['diagnosticoprincipal'].nunique(dropna=False),
        df_comorbilidad['d1_limpio'].nunique(dropna=False),
        df_comorbilidad['cie101'].nunique(dropna=False)
    ]
})
print("___ 1. RESUMEN DE CARDINALIDAD (D1) ___")
display(resumen_cardinalidad)

# 3. Análisis por Código CIE-10 - Frecuencia de Registros
print("--- 2. FRECUENCIA DE REGISTROS POR CÓDIGO CIE-10  ---")
resumen_cie10 = df_comorbilidad['cie101'].value_counts(normalize=True).to_frame(name='Proporción')
resumen_cie10['Numero de Registros'] = df_comorbilidad['cie101'].value_counts()
display(resumen_cie10.style.format({'Proporción': '{:.2%}'}))

# 4. Análisis de Inconsistencia: Variantes por Código CIE-10
# Comparamos cuántas descripciones 'sucias' vs 'limpias' hay por código
inconsistencia = df_comorbilidad.groupby('cie101').agg(
    Descripciones_Originales_Valores_Unicos=('diagnosticoprincipal', 'nunique'),
    Descripciones_Limpias_Valores_Unicos=('d1_limpio', 'nunique')
).sort_values('Descripciones_Originales_Valores_Unicos', ascending=False)

print("\n___ 3. INCONSISTENCIA SINTÁCTICA POR CÓDIGO CIE-10 ___")
display(inconsistencia)

# 5. La variante más frecuente por cada código (La "Moda" de cada diagnóstico)
print("\n--- 4. DESCRIPCIÓN MÁS FRECUENTE PARA CADA UNO DE LOS 3 CÓDIGOS ---")
moda_por_cie = df_comorbilidad.groupby('cie101')['d1_limpio'].agg(lambda x: x.mode()[0]).to_frame(name='Descripción más frecuente')
moda_por_cie['Casos'] = df_comorbilidad.groupby('cie101')['d1_limpio'].count()
moda_por_cie = moda_por_cie.sort_values('Casos', ascending=False)
display(moda_por_cie)

# 6. Top 10 Variaciones en la captura textual de diagnósticos vs códigos CIE-10
codigo_a_analizar = 'U07.1'  # <--- Cambia para analizar U07.2 o U09.9

print(f"\n___ 5. TOP 10 VARIACIONES TEXTUALES PARA EL CÓDIGO: {codigo_a_analizar} ___")
top_variaciones = df_comorbilidad[df_comorbilidad['cie101'] == codigo_a_analizar]['d1_limpio'].value_counts().to_frame(name=f'Frecuencia codigo: {codigo_a_analizar}')

display(top_variaciones.head(10))


___ 1. RESUMEN DE CARDINALIDAD (D1) ___


,Campo,Valores Únicos
0,diagnosticoprincipal (Original),214
1,diagnosticoprincipal (Limpio),176
2,cie101,3


--- 2. FRECUENCIA DE REGISTROS POR CÓDIGO CIE-10  ---


,Proporción,Numero de Registros
cie101,,
U07.1,98.57%,4217
U07.2,1.36%,58
U09.9,0.07%,3



___ 3. INCONSISTENCIA SINTÁCTICA POR CÓDIGO CIE-10 ___


,Descripciones_Originales_Valores_Unicos,Descripciones_Limpias_Valores_Unicos
cie101,,
U07.1,184,146
U07.2,30,30
U09.9,2,2



--- 4. DESCRIPCIÓN MÁS FRECUENTE PARA CADA UNO DE LOS 3 CÓDIGOS ---


,Descripción más frecuente,Casos
cie101,,
U07.1,NEUMONIA POR SARS COV2,4217
U07.2,NEUMONIA POR PROBABLE SARS COV2,58
U09.9,POST COVID,3



___ 5. TOP 10 VARIACIONES TEXTUALES PARA EL CÓDIGO: U07.1 ___


,Frecuencia codigo: U07.1
d1_limpio,
NEUMONIA POR SARS COV2,1430
SARS-COV2,490
COVID 19,355
ENFERMEDAD RESPIRATORIA AGUDA POR NCOV,314
NEUMONIA POR COVID 19,254
INFECCION POR SARS COV2,140
NEUMONIA VIRAL POR COVID 19,125
NEUMONIA VIRAL POR SARS COV2,101
NEUMONIA POR NCOV,91


### 2.2 Columnas relacionadas con el **segundo diagnóstico** del paciente:

In [18]:
# --- ANÁLISIS CONSOLIDADO: SEGUNDO DIAGNÓSTICO (2) ---
# 1. Limpieza básica para el análisis
from ast import mod


df_comorbilidad['d2_limpio'] = df_comorbilidad['diagnostico2'].str.upper().str.strip()

# 2. Resumen de Cardinalidad y Redundancia (Incluye dx2)
resumen_cardinalidad_2 = pd.DataFrame({
    'Campo': ['diagnostico2 (Original)', 'diagnostico2 (Limpio)', 'cie102', 'dx2'],
    'Valores Únicos': [
        df_comorbilidad['diagnostico2'].nunique(dropna=False),
        df_comorbilidad['d2_limpio'].nunique(dropna=False),
        df_comorbilidad['cie102'].nunique(dropna=False),
        df_comorbilidad['dx2'].nunique(dropna=False)
    ]
})
print("___ 1. RESUMEN DE CARDINALIDAD (D2) ___")
display(resumen_cardinalidad_2)

# 3. Análisis por Código CIE-10 (Top 10 patologías secundarias)
print("\n--- 2. DISTRIBUCIÓN POR CÓDIGO CIE-10 (D2) ---")
resumen_cie10_2 = df_comorbilidad['cie102'].value_counts(normalize=True).to_frame(name='Proporción')
resumen_cie10_2['Casos'] = df_comorbilidad['cie102'].value_counts()
display(resumen_cie10_2.head(10).style.format({'Proporción': '{:.2%}'}))

# 4. Análisis de Inconsistencia: Variantes por Código CIE-10
inconsistencia_2 = df_comorbilidad.groupby('cie102').agg(
    Variantes_Originales=('diagnostico2', 'nunique'),
    Variantes_Limpias=('d2_limpio', 'nunique'),
    Codigos_dx2_asociados=('dx2', 'nunique') # Para confirmar redundancia 1:1
).sort_values('Variantes_Originales', ascending=False)

print("\n___ 3. INCONSISTENCIA SINTÁCTICA Y VALIDACIÓN DX2 ___")
display(inconsistencia_2.head(10))

# 5. La variante más frecuente por cada código (Moda)
print("\n--- 4. DESCRIPCIÓN MÁS FRECUENTE PARA CADA UNO DE LOS CÓDIGOS ---")
moda_por_cie_2 = df_comorbilidad.groupby('cie102')['d2_limpio'].agg(lambda x: x.mode()[0] if not x.mode().empty else "N/A").to_frame(name='Descripción más frecuente')
moda_por_cie_2['Casos'] = df_comorbilidad.groupby('cie102')['d2_limpio'].count()
moda_por_cie_2 = moda_por_cie_2.sort_values('Casos', ascending=False)
display(moda_por_cie_2.head(10))


# 6. Top 10 Variaciones para un código específico (Similitud con D1)
codigo_a_analizar_2 = 'J96.0' # Falla respiratoria aguda (el más común en D2)

print(f"\n___ 5. TOP 10 VARIACIONES TEXTUALES PARA EL CÓDIGO: {codigo_a_analizar_2} ___")
top_variaciones_2 = df_comorbilidad[df_comorbilidad['cie102'] == codigo_a_analizar_2]['d2_limpio'].value_counts().to_frame(name=f'Frecuencia ({codigo_a_analizar_2})')
display(top_variaciones_2.head(10))


___ 1. RESUMEN DE CARDINALIDAD (D2) ___


,Campo,Valores Únicos
0,diagnostico2 (Original),205
1,diagnostico2 (Limpio),196
2,cie102,115
3,dx2,115



--- 2. DISTRIBUCIÓN POR CÓDIGO CIE-10 (D2) ---


,Proporción,Casos
cie102,,
J96.0,53.62%,1563
I10.X,11.90%,347
E66.9,5.39%,157
E11.9,5.35%,156
A41.9,2.71%,79
E66.8,2.23%,65
J39.8,1.85%,54
J18.9,0.99%,29
J84.9,0.89%,26



___ 3. INCONSISTENCIA SINTÁCTICA Y VALIDACIÓN DX2 ___


,Variantes_Originales,Variantes_Limpias,Codigos_dx2_asociados
cie102,,,
J96.0,24,22,1
J18.9,8,8,1
J84.9,6,6,1
J98.4,6,6,1
E66.9,6,5,1
E66.8,5,5,1
E11.9,4,4,1
C34.9,4,3,1
B23.8,3,3,1



--- 4. DESCRIPCIÓN MÁS FRECUENTE PARA CADA UNO DE LOS CÓDIGOS ---


,Descripción más frecuente,Casos
cie102,,
J96.0,SINDROME DE INSUFICIENCIA RESPIRATORIA AGUDA,1563
I10.X,HAS,347
E66.9,OBESIDAD GRADO I,157
E11.9,DIABETES MELLITUS TIPO II,156
A41.9,CHOQUE SEPTICO,79
E66.8,OBESIDAD GRADO II,65
J39.8,ESTENOSIS TRAQUEAL,54
J18.9,NEUMONIA ASOCIADA A VENTILACION MECANICA,29
J84.9,ENFERMEDAD PULMONAR INTERSTICIAL DIFUSA,26



___ 5. TOP 10 VARIACIONES TEXTUALES PARA EL CÓDIGO: J96.0 ___


,Frecuencia (J96.0)
d2_limpio,
SINDROME DE INSUFICIENCIA RESPIRATORIA AGUDA,1121
INSUFICIENCIA RESPIRATORIA TIPO I,214
INSUFICIENCIA RESPIRATORIA AGUDA,171
SINDROME DE INSUFICIENCIA RESPIRATORIAS AGUDA,17
SINDROME DE INSUFICIENCAI RESPIRATORIA AGUDA,11
INSUFICENCIA RESPIRATORIA AGUDA,4
SINDROME DE INSUFIOCIENCIA RESPIRATORIA AGUDA,3
INFECCION RESPIRATORIA AGUDA,3
INSUFICIENCIA RESPIRATORIA AGUDA TIPO I,3


### 2.3 Columnas relacionadas con el **tercer diagnóstico** del paciente:

In [20]:
# 1. Limpieza básica para el análisis
df_comorbilidad['d3_limpio'] = df_comorbilidad['diagnostico3'].str.upper().str.strip()

# 2. Resumen de Cardinalidad y Redundancia
resumen_cardinalidad_3 = pd.DataFrame({
    'Campo': ['diagnostico3 (Original)', 'diagnostico3 (Limpio)', 'cie103', 'dx3'],
    'Valores Únicos': [
        df_comorbilidad['diagnostico3'].nunique(dropna=False),
        df_comorbilidad['d3_limpio'].nunique(dropna=False),
        df_comorbilidad['cie103'].nunique(dropna=False),
        df_comorbilidad['dx3'].nunique(dropna=False)
    ]
})
print("___ 1. RESUMEN DE CARDINALIDAD (D3) ___")
display(resumen_cardinalidad_3)

# 3. Distribución por Código CIE-10 (Principales patologías en D3)
print("\n--- 2. DISTRIBUCIÓN POR CÓDIGO CIE-10 (D3) ---")
resumen_cie10_3 = df_comorbilidad['cie103'].value_counts(normalize=True).to_frame(name='Proporción')
resumen_cie10_3['Casos'] = df_comorbilidad['cie103'].value_counts()
display(resumen_cie10_3.head(10).style.format({'Proporción': '{:.2%}'}))

# 4. Análisis de Inconsistencia Sintáctica
inconsistencia_3 = df_comorbilidad.groupby('cie103').agg(
    Variantes_Originales=('diagnostico3', 'nunique'),
    Variantes_Limpias=('d3_limpio', 'nunique'),
    Codigos_dx3_asociados=('dx3', 'nunique')
).sort_values('Variantes_Originales', ascending=False)

print("\n___ 3. INCONSISTENCIA SINTÁCTICA Y VALIDACIÓN DX3 ___")
display(inconsistencia_3.head(10))

# 5. Descripción más frecuente (Moda) por cada código
print("\n--- 4. DESCRIPCIÓN RECOMENDADA (MODA) POR CÓDIGO ---")
moda_por_cie_3 = df_comorbilidad.groupby('cie103')['d3_limpio'].agg(lambda x: x.mode()[0] if not x.mode().empty else "N/A").to_frame(name='Descripción más frecuente')
moda_por_cie_3['Casos'] = df_comorbilidad.groupby('cie103')['d3_limpio'].count()
moda_por_cie_3 = moda_por_cie_3.sort_values('Casos', ascending=False)
display(moda_por_cie_3.head(10))

# 6. Variaciones de un código representativo en D3
# Nota: 'E11.9' suele ser común en diagnósticos secundarios (Diabetes)
codigo_a_analizar_3 = 'E11.9'  # Diabetes mellitus tipo 2 sin complicaciones HTAS resultó ser el más frecuente en D3, pero sin variaciones en los diagnósticos escritos.

if codigo_a_analizar_3 in df_comorbilidad['cie103'].values:
    print(f"\n___ 5. TOP VARIACIONES TEXTUALES PARA EL CÓDIGO: {codigo_a_analizar_3} ___")
    top_variaciones_3 = df_comorbilidad[df_comorbilidad['cie103'] == codigo_a_analizar_3]['d3_limpio'].value_counts().to_frame(name=f'Frecuencia ({codigo_a_analizar_3})')
    display(top_variaciones_3.head(10))


___ 1. RESUMEN DE CARDINALIDAD (D3) ___


,Campo,Valores Únicos
0,diagnostico3 (Original),84
1,diagnostico3 (Limpio),83
2,cie103,60
3,dx3,60



--- 2. DISTRIBUCIÓN POR CÓDIGO CIE-10 (D3) ---


,Proporción,Casos
cie103,,
I10.X,27.67%,432
E11.9,23.83%,372
E66.9,16.21%,253
A41.9,11.98%,187
E66.8,5.00%,78
J96.0,4.68%,73
J18.9,0.90%,14
M06.9,0.77%,12
J93.9,0.70%,11



___ 3. INCONSISTENCIA SINTÁCTICA Y VALIDACIÓN DX3 ___


,Variantes_Originales,Variantes_Limpias,Codigos_dx3_asociados
cie103,,,
E66.9,8,7,1
J96.0,5,5,1
E66.8,4,4,1
J93.9,4,4,1
E11.9,4,4,1
A41.9,3,3,1
I26.9,3,3,1
K72.9,2,2,1
K92.2,2,2,1



--- 4. DESCRIPCIÓN RECOMENDADA (MODA) POR CÓDIGO ---


,Descripción más frecuente,Casos
cie103,,
I10.X,HAS,432
E11.9,DIABETES MELLITUS TIPO II,372
E66.9,SOBREPESO,253
A41.9,CHOQUE SEPTICO,187
E66.8,OBESIDAD GRADO II,78
J96.0,SINDROME DE INSUFICIENCIA RESPIRATORIA AGUDA,73
J18.9,NEUMONIA ASOCIADA A VENTILACION MECANICA,14
M06.9,ARTRITIS REUMATOIDE,12
J93.9,NEUMOTORAX,11



___ 5. TOP VARIACIONES TEXTUALES PARA EL CÓDIGO: E11.9 ___


,Frecuencia (E11.9)
d3_limpio,
DIABETES MELLITUS TIPO II,354
DIABETES MLEELITUS TIPO II,15
DM2,2
DIABETES MELLITUS TIPO II,1


In [66]:
cols_diagnostico_3 = ['diagnostico3', 'cie103', 'dx3']

# Resumen de valores únicos en formato tabla
resumen_unicos = pd.DataFrame({
    'Campo': cols_diagnostico_3,
    'Valores Únicos': [df_comorbilidad[c].nunique(dropna=False) for c in cols_diagnostico_3]
})
print("--- Resumen de Cardinalidad ---")
display(resumen_unicos)

# Análisis de consistencia: ¿Cuántas descripciones textuales tiene cada código CIE10?
# Para intentar explicar por qué 'diagnostico3' tiene 84 valores mientras que 'cie103' tiene 60
consistencia_diagnostico3 = df_comorbilidad.groupby('cie103')['diagnostico3'].nunique(dropna=False).sort_values(ascending=False).to_frame(name='Descripciones distintas')
print("\n--- Códigos con múltiples descripciones (Inconsistencias o Variaciones) ---")
display(consistencia_diagnostico3.head(5))

# Análisis de consistencia: ¿Cuántos códigos dx3 tiene cada código CIE10? Hay relación 1 a 1 o hay múltiples códigos dx3 asociados a un mismo código CIE10?
consistencia_dx3 = df_comorbilidad.groupby('cie103')['dx3'].nunique(dropna=False).sort_values(ascending=False).to_frame(name='Códigos cie103 VS dx3 distintos')
print("\n--- Códigos CIE-103 con múltiples códigos dx3 ---")
display(consistencia_dx3.head(5))

# Top 10 con códigos y descripciones más frecuentes
print("\n--- Top 10 Diagnósticos Frecuentes ---")
df_comorbilidad.groupby(['cie103', 'diagnostico3']).size().reset_index(name='Frecuencia').sort_values('Frecuencia', ascending=False).head(10)


--- Resumen de Cardinalidad ---


,Campo,Valores Únicos
0,diagnostico3,84
1,cie103,60
2,dx3,60



--- Códigos con múltiples descripciones (Inconsistencias o Variaciones) ---


,Descripciones distintas
cie103,
E66.9,8
J96.0,5
E66.8,4
J93.9,4
E11.9,4



--- Códigos CIE-103 con múltiples códigos dx3 ---


,Códigos cie103 VS dx3 distintos
cie103,
A04.7,1
A16.2,1
A41.8,1
A41.9,1
A49.8,1



--- Top 10 Diagnósticos Frecuentes ---


,cie103,diagnostico3,Frecuencia
31,I10.X,HAS,432
13,E11.9,DIABETES MELLITUS TIPO II,354
4,A41.9,CHOQUE SEPTICO,185
27,E66.9,SOBREPESO,115
24,E66.9,OBESIDAD GRADO I,108
17,E66.8,OBESIDAD GRADO II,50
65,J96.0,SINDROME DE INSUFICIENCIA RESPIRATORIA AGUDA,36
63,J96.0,INSUFICIENCIA RESPIRATORIA TIPO I,22
22,E66.9,OBESIDAD,22
14,E11.9,DIABETES MLEELITUS TIPO II,15


### 2.4 Columnas relacionadas con el **cuarto diagnóstico** del paciente:

In [21]:
# 1. Limpieza básica para el análisis
df_comorbilidad['d4_limpio'] = df_comorbilidad['diagnostico4'].str.upper().str.strip()

# 2. Resumen de Cardinalidad y Redundancia
resumen_cardinalidad_4 = pd.DataFrame({
    'Campo': ['diagnostico4 (Original)', 'diagnostico4 (Limpio)', 'cie104', 'dx4'],
    'Valores Únicos': [
        df_comorbilidad['diagnostico4'].nunique(dropna=False),
        df_comorbilidad['d4_limpio'].nunique(dropna=False),
        df_comorbilidad['cie104'].nunique(dropna=False),
        df_comorbilidad['dx4'].nunique(dropna=False)
    ]
})
print("___ 1. RESUMEN DE CARDINALIDAD (D4) ___")
display(resumen_cardinalidad_4)

# 3. Distribución por Código CIE-10 (Principales patologías en D4)
print("\n--- 2. DISTRIBUCIÓN POR CÓDIGO CIE-10 (D4) ---")
resumen_cie10_4 = df_comorbilidad['cie104'].value_counts(normalize=True).to_frame(name='Proporción')
resumen_cie10_4['Casos'] = df_comorbilidad['cie104'].value_counts()
display(resumen_cie10_4.head(10).style.format({'Proporción': '{:.2%}'}))

# 4. Análisis de Inconsistencia Sintáctica
inconsistencia_4 = df_comorbilidad.groupby('cie104').agg(
    Variantes_Originales=('diagnostico4', 'nunique'),
    Variantes_Limpias=('d4_limpio', 'nunique'),
    Codigos_dx4_asociados=('dx4', 'nunique')
).sort_values('Variantes_Originales', ascending=False)

print("\n___ 3. INCONSISTENCIA SINTÁCTICA Y VALIDACIÓN DX4 ___")
display(inconsistencia_4.head(10))

# 5. Descripción más frecuente (Moda) por cada código
print("\n--- 4. DESCRIPCIÓN RECOMENDADA (MODA) POR CÓDIGO ---")
moda_por_cie_4 = df_comorbilidad.groupby('cie104')['d4_limpio'].agg(lambda x: x.mode()[0] if not x.mode().empty else "N/A").to_frame(name='Descripción más frecuente')
moda_por_cie_4['Casos'] = df_comorbilidad.groupby('cie104')['d4_limpio'].count()
moda_por_cie_4 = moda_por_cie_4.sort_values('Casos', ascending=False)
display(moda_por_cie_4.head(10))

# 6. Variaciones de un código representativo en D4
codigo_a_analizar_4 = 'E11.9'  # Diabetes mellitus tipo 2 resultó ser el más frecuente en D4, pero sin variaciones en los diagnósticos escritos.

if codigo_a_analizar_4 in df_comorbilidad['cie104'].values:
    print(f"\n___ 5. TOP VARIACIONES TEXTUALES PARA EL CÓDIGO: {codigo_a_analizar_4} ___")
    top_variaciones_4 = df_comorbilidad[df_comorbilidad['cie104'] == codigo_a_analizar_4]['d4_limpio'].value_counts().to_frame(name=f'Frecuencia ({codigo_a_analizar_4})')
    display(top_variaciones_4.head(10))


___ 1. RESUMEN DE CARDINALIDAD (D4) ___


,Campo,Valores Únicos
0,diagnostico4 (Original),25
1,diagnostico4 (Limpio),24
2,cie104,15
3,dx4,15



--- 2. DISTRIBUCIÓN POR CÓDIGO CIE-10 (D4) ---


,Proporción,Casos
cie104,,
E11.9,45.77%,211
A41.9,18.66%,86
E66.9,16.70%,77
E66.8,10.63%,49
I10.X,4.99%,23
J96.0,0.87%,4
M06.9,0.65%,3
J44.9,0.43%,2
K92.1,0.22%,1



___ 3. INCONSISTENCIA SINTÁCTICA Y VALIDACIÓN DX4 ___


,Variantes_Originales,Variantes_Limpias,Codigos_dx4_asociados
cie104,,,
E66.9,5,4,1
E66.8,5,5,1
E11.9,3,3,1
J96.0,2,2,1
G47.3,1,1,1
A41.9,1,1,1
I10.X,1,1,1
I25.9,1,1,1
J44.9,1,1,1



--- 4. DESCRIPCIÓN RECOMENDADA (MODA) POR CÓDIGO ---


,Descripción más frecuente,Casos
cie104,,
E11.9,DIABETES MELLITUS TIPO II,211
A41.9,CHOQUE SEPTICO,86
E66.9,OBESIDAD GRADO I,77
E66.8,OBESIDAD GRADO III,49
I10.X,HAS,23
J96.0,INSUFICIENCIA RESPIRATORIA TIPO I,4
M06.9,ARTRITIS REUMATOIDE,3
J44.9,EPOC,2
G47.3,SAOS,1



___ 5. TOP VARIACIONES TEXTUALES PARA EL CÓDIGO: E11.9 ___


,Frecuencia (E11.9)
d4_limpio,
DIABETES MELLITUS TIPO II,207
DIAABTES MELLITUS TIPO II,2
DIABTES MELLITUS TIPO II,2


In [67]:
cols_diagnostico_4 = ['diagnostico4', 'cie104', 'dx4']

# Resumen de valores únicos en formato tabla
resumen_unicos = pd.DataFrame({
    'Campo': cols_diagnostico_4,
    'Valores Únicos': [df_comorbilidad[c].nunique(dropna=False) for c in cols_diagnostico_4]
})
print("--- Resumen de Cardinalidad ---")
display(resumen_unicos)

# Análisis de consistencia: ¿Cuántas descripciones textuales tiene cada código CIE10?
# Para intentar explicar por qué 'diagnostico4' tiene 175 valores mientras que 'cie104' tiene 105
consistencia_diagnostico4 = df_comorbilidad.groupby('cie104')['diagnostico4'].nunique(dropna=False).sort_values(ascending=False).to_frame(name='Descripciones distintas')
print("\n--- Códigos con múltiples descripciones (Inconsistencias o Variaciones) ---")
display(consistencia_diagnostico4.head(5))

# Análisis de consistencia: ¿Cuántos códigos dx4 tiene cada código CIE10? Hay relación 1 a 1 o hay múltiples códigos dx4 asociados a un mismo código CIE10?
consistencia_dx4 = df_comorbilidad.groupby('cie104')['dx4'].nunique(dropna=False).sort_values(ascending=False).to_frame(name='Códigos cie104 VS dx4 distintos')
print("\n--- Códigos CIE-104 con múltiples códigos dx4 ---")
display(consistencia_dx4.head(5))

# Top 10 con códigos y descripciones más frecuentes
print("\n--- Top 10 Diagnósticos Frecuentes ---")
df_comorbilidad.groupby(['cie104', 'diagnostico4']).size().reset_index(name='Frecuencia').sort_values('Frecuencia', ascending=False).head(10)


--- Resumen de Cardinalidad ---


,Campo,Valores Únicos
0,diagnostico4,25
1,cie104,15
2,dx4,15



--- Códigos con múltiples descripciones (Inconsistencias o Variaciones) ---


,Descripciones distintas
cie104,
E66.9,5
E66.8,5
E11.9,3
J96.0,2
G47.3,1



--- Códigos CIE-104 con múltiples códigos dx4 ---


,Códigos cie104 VS dx4 distintos
cie104,
A41.9,1
E11.9,1
E66.8,1
E66.9,1
G47.3,1



--- Top 10 Diagnósticos Frecuentes ---


,cie104,diagnostico4,Frecuencia
2,E11.9,DIABETES MELLITUS TIPO II,207
0,A41.9,CHOQUE SEPTICO,86
11,E66.9,OBESIDAD GRADO I,36
13,E66.9,SOBREPESO,30
6,E66.8,OBESIDAD GRADO III,23
15,I10.X,HAS,23
5,E66.8,OBESIDAD GRADO II,18
9,E66.9,OBESIDAD,8
7,E66.8,OBESIDAD MORBIDA,5
24,M06.9,ARTRITIS REUMATOIDE,3


### 2.5 Análisis de columnas relacionadas con **comorbilidades**

In [ ]:
print("Valores únicos en la columna 'comorbi':", len(df_comorbilidad['comorbi'].unique()))
df_comorbilidad['comorbi'].value_counts().to_frame()

Valores únicos en la columna 'comorbi' 10


,count
comorbi,
Ninguna,1363
Otras,1214
Obesidad/Trastornos alimentación,675
Diabetes Mellitus,605
HTAS,327
Enfermedades del sistema renal y urinario,42
TEP &/| HAP,26
Cardiopatía Isquémica,15
EAP & ERGE,9


In [ ]:
print("Valores únicos en la columna 'comorbicv':", len(df_comorbilidad['comorbicv'].unique()))
df_comorbilidad['comorbicv'].value_counts().to_frame()

Valores únicos en la columna 'comorbicv' 5


,count
comorbicv,
0.0,3432
HTAS,789
TEP &/| HAP,28
Cardiopatía Isquémica,26
Insuficiencia Cardíaca,3


## 3. ..._TrabajoSocial.csv 
**Información sobre el trabajo social de los pacientes:**

In [6]:
df_trabajo_social = pd.read_csv(ruta_datos / archivo_3)
df_trabajo_social.info(memory_usage=False)

<class 'pandas.DataFrame'>
RangeIndex: 14796 entries, 0 to 14795
Data columns (total 20 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   AÑO                                14796 non-null  int64
 1   FILA                               14796 non-null  int64
 2   EXPEDIENTE                         14796 non-null  int64
 3   NO. HISTORIA                       14796 non-null  str  
 4   FECHA DE ELABORACIÓN               14796 non-null  str  
 5   APELLIDO PATERNO                   14795 non-null  str  
 6   APELLIDO MATERNO                   14697 non-null  str  
 7   NOMBRE                             14777 non-null  str  
 8   EDAD                               14145 non-null  str  
 9   FECHA DE NACIMIENTO                14145 non-null  str  
 10  GENERO                             14128 non-null  str  
 11  DIAGNOSTICO                        5726 non-null   str  
 12  ESCOLARIDAD                  

## 4. Duplicados

### 4.0 Análisis ingenuo de duplicados utilizando `df.duplicated()`

In [ ]:
dfs = {
    'CostoPacientes_Econo': df_economico,
    'Pacientes_DiagnosticoComorbilidad': df_comorbilidad,
    'TrabajoSocial': df_trabajo_social
}

resumen_duplicados = []

for nombre, df in dfs.items():
    total_filas = len(df)
    duplicados = df.duplicated().sum()
    proporcion = (duplicados / total_filas)

    resumen_duplicados.append({
        'Dataset': nombre,
        'Total Filas': total_filas,
        'Filas Duplicadas': duplicados,
        'Proporción %': proporcion
    })

# Convertimos a DataFrame para una visualización limpia
df_resumen_duplicados = pd.DataFrame(resumen_duplicados)

print("___ ANÁLISIS GLOBAL DE DUPLICADOS ___")
display(df_resumen_duplicados.style.format({'Proporción %': '{:.2%}'}))

# Ejemplo detallado si existen duplicados en la base de comorbilidad
if df_comorbilidad.duplicated().any():
    print(f"\n[!] Muestra de filas duplicadas en DiagnosticoComorbilidad:")
    display(df_comorbilidad[df_comorbilidad.duplicated(keep=False)].sort_values(by=list(df_comorbilidad.columns[:3])).head(6))
else:
    print("\n[OK] No se encontraron filas idénticas en DiagnosticoComorbilidad.")


___ ANÁLISIS GLOBAL DE DUPLICADOS (Utilizando simplemente df.duplicated())___


,Dataset,Total Filas,Filas Duplicadas,Proporción %
0,CostoPacientes_Econo,4632,1,0.02%
1,Pacientes_DiagnosticoComorbilidad,4278,0,0.00%
2,TrabajoSocial,14796,0,0.00%



[OK] No se encontraron filas idénticas en DiagnosticoComorbilidad.


### 4.1 Análisis global de duplicados inspeccionando por columnas clave 

In [9]:

# Análisis de duplicados por Llave Compuesta (Paciente + Fecha Ingreso)
print("___ ANÁLISIS DE DUPLICADOS POR LLAVE PACIENTE ___")

# Para df_economico: llave = NOMBRE_DEL_PACIENTE + FECHA_INGRESO_INER
dup_pacientes_econ = df_economico.duplicated(subset=['NOMBRE_DEL_PACIENTE', 'FECHA_INGRESO_INER'], keep=False).sum()
print(f"Registros con mismo Nombre y Fecha de Ingreso en CostoPacientes_Econo: {dup_pacientes_econ}")

# Para df_comorbilidad: llave = nombre + fechaing
dup_pacientes_comor = df_comorbilidad.duplicated(subset=['nombre', 'fechaing'], keep=False).sum()
print(f"Registros con mismo Nombre y Fecha de Ingreso en DiagnosticoComorbilidad: {dup_pacientes_comor}")

if dup_pacientes_econ > 0:
    display(df_economico[df_economico.duplicated(subset=['NOMBRE_DEL_PACIENTE', 'FECHA_INGRESO_INER'], keep=False)].sort_values(by='NOMBRE_DEL_PACIENTE'))

if dup_pacientes_comor > 0:
    display(df_comorbilidad[df_comorbilidad.duplicated(subset=['nombre', 'fechaing'], keep=False)].sort_values(by='nombre'))

___ ANÁLISIS DE DUPLICADOS POR LLAVE PACIENTE ___
Registros con mismo Nombre y Fecha de Ingreso en CostoPacientes_Econo: 20
Registros con mismo Nombre y Fecha de Ingreso en DiagnosticoComorbilidad: 6


,EXP,NOMBRE_DEL_PACIENTE,SEXO,EDAD,GRUPO_EDAD,RESULTADO,ETIQUETAS_COVID,MOTIVO_DE_EGRESO,FECHA_INGRESO_INER,FECHA_DE_ALTA_MEJORIA,...,TOTAL_DE_EGRESOS,ESCOLARIDAD,OCUPACION,DERECHOHABIENTE_Y/O_BENEFICIARIO,VULNERABILIDAD_SOCIOECONOMICA,NIVEL_SOCIOECONOMICO,ESTADO_RESIDENCIA,CLAVE_GEOESTADISTICA_ESTATAL,MUNICIPIO_RESIDENCIA,CLAVE_GEOESTADISTICA_MUNICIPAL
385,242544,ALVAREZ RICO GABRIEL,M,36.0,35-39,SARS-COV2,COVID,ALTA POR MEJORIA,2022-02-11,2022-02-13,...,NaN,NaN,NaN,IMSS,False,2,CIUDAD DE MEXICO,9.0,GUSTAVO A. MADERO,5.0
456,242544,ALVAREZ RICO GABRIEL,M,36.0,35-39,SARS-COV2,COVID,ALTA POR MEJORIA,2022-02-11,2022-02-13,...,NaN,NaN,NaN,IMSS,False,2,CIUDAD DE MEXICO,9.0,GUSTAVO A. MADERO,5.0
2726,241414,ARGUELLO DE ROSINI ANITA ESTER,F,71.0,70-74,SARS-COV2,COVID,ALTA POR MEJORIA,2021-10-16,2021-10-28,...,9000.0,SECUNDARIA,NaN,NINGUNO,False,2E,CIUDAD DE MEXICO,9.0,COYOACAN,3.0
2777,241414,ARGUELLO DE ROSINI ANITA ESTER,F,71.0,70-74,SARS-COV2,COVID,ALTA POR MEJORIA,2021-10-16,2021-10-28,...,NaN,NaN,NaN,NINGUNO,False,NaN,CIUDAD DE MEXICO,9.0,COYOACAN,3.0
4202,238879,CLEMENTE MILLAN MARIA ALEJANDRA,F,49.0,45-49,NaN,NaN,ALTA POR MEJORIA,2020-11-08,2020-11-21,...,5912.0,SECUNDARIA,HOGAR,IMSS,True,3,CIUDAD DE MEXICO,9.0,IZTAPALAPA,7.0
4469,238879,CLEMENTE MILLAN MARIA ALEJANDRA,F,49.0,45-49,NaN,NaN,ALTA POR MEJORIA,2020-11-08,2020-12-04,...,5912.0,SECUNDARIA,HOGAR,IMSS,True,3,CIUDAD DE MEXICO,9.0,IZTAPALAPA,7.0
1964,240439,FLORES GUTIERREZ DIEGO GONZALO,M,36.0,35-39,SARS-COV2,NaN,ALTA POR MEJORIA,2021-06-16,2021-06-18,...,21900.0,NMS,NaN,NINGUNO,True,3,CIUDAD DE MEXICO,9.0,CUAUHTEMOC,15.0
2103,240439,FLORES GUTIERREZ DIEGO GONZALO,M,36.0,35-39,SARS-COV2,NaN,ALTA POR MEJORIA,2021-06-16,2021-07-13,...,21900.0,NMS,NaN,NINGUNO,True,3,CIUDAD DE MEXICO,9.0,CUAUHTEMOC,15.0
3121,237721,GARCIA VAZQUEZ PATRICIA,F,46.0,45-49,SARS-COV2,NaN,ALTA POR MEJORIA,2020-05-09,2020-05-15,...,8140.0,SECUNDARIA,HOGAR,NINGUNO,True,2,CIUDAD DE MEXICO,9.0,COYOACAN,3.0
3372,237721,GARCIA VAZQUEZ PATRICIA,F,46.0,45-49,SARS-COV2,NaN,ALTA POR MEJORIA,2020-05-09,2020-06-30,...,8140.0,SECUNDARIA,HOGAR,NINGUNO,True,2,CIUDAD DE MEXICO,9.0,COYOACAN,3.0


,expediente,nombre,fechaing,fechaegr,diagnosticoprincipal,cie101,diagnostico2,cie102,diagnostico3,cie103,...,dx4,obesidad,obesidad1,cardiopatia,comorbi,diabetes,nefropatia,eaperge,tephap,comorbicv
1543,239273,CARLOS PONCE CHAVEZ,2020-12-30,2021-01-05,NEUMONIA POR SARS COV2,U07.1,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,Ninguna,0.0,0.0,0.0,0.0,0.0
1887,239273,CARLOS PONCE CHAVEZ,2020-12-30,2021-02-23,SARS-COV2,U07.1,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,Ninguna,0.0,0.0,0.0,0.0,0.0
2068,239856,JUAN VICTOR NU?EZ JUAREZ,2021-03-14,2021-03-17,POST COVID,U07.1,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,Ninguna,0.0,0.0,0.0,0.0,0.0
2180,239856,JUAN VICTOR NU?EZ JUAREZ,2021-03-14,2021-04-01,NEUMONIA POR SARS COV2,U07.1,INSUFICIENCIA RESPIRATORIA TIPO I,J96.0,HAS,I10.X,...,E11.9,0.0,0.0,1.0,Diabetes Mellitus,1.0,0.0,0.0,0.0,HTAS
4113,249200,VICENTE MARTIN VALENCIA CHAVEZ,2023-03-04,2023-03-04,COVID 19,U07.1,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,Ninguna,0.0,0.0,0.0,0.0,0.0
4115,249200,VICENTE MARTIN VALENCIA CHAVEZ,2023-03-04,2023-03-06,COVID 19,U07.1,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,Ninguna,0.0,0.0,0.0,0.0,0.0


**Hallazgo:** 

Efectivamente se detectaron 

* 20 duplicados en la base de comorbilidad
* 6 duplicados en la base económica
* 4 duplicados en la base de trabajo social

Actualmente no se muestra, pero también, sería importante mostrar la cantidad de veces que se repite cada registro para entender mejor la magnitud del problema de duplicados.

In [23]:
# --- ANÁLISIS REFINADO DE DUPLICADOS POR LLAVE LÓGICA (PACIENTE + EVENTO) ---

print("___ ANÁLISIS DE DUPLICADOS POR PACIENTE ___\n")

def normalizar_llave(valor):
    """Normaliza espacios en un valor de llave para comparación robusta."""
    if pd.isna(valor):
        return valor
    return " ".join(str(valor).strip().split())

def analizar_duplicados(df, llave, nombre_dataset):
    """Analiza duplicados por llave lógica normalizada, mostrando solo columnas con diferencias."""
    df_tmp = df.copy()
    # Normalizar espacios en columnas de llave (tipo string)
    llave_norm = []
    for col in llave:
        if df_tmp[col].dtype == 'object':
            col_norm = f'_llave_{col}'
            df_tmp[col_norm] = df_tmp[col].apply(normalizar_llave)
            llave_norm.append(col_norm)
        else:
            llave_norm.append(col)

    mask_dup = df_tmp.duplicated(subset=llave_norm, keep=False)
    n_dup = mask_dup.sum()
    print(f"{nombre_dataset}: {n_dup} registros duplicados por llave ({' + '.join(llave)})")

    if n_dup == 0:
        return

    df_dup = df_tmp[mask_dup].copy()
    df_dup['fila_csv'] = df_dup.index + 2  # +1 header + 1 por índice 0-based

    # Distribución de repeticiones por grupo
    conteo = df_dup.groupby(llave_norm, dropna=False).size().value_counts().sort_index()
    print("   Distribución de repeticiones:")
    for n_reps, n_grupos in conteo.items():
        print(f"   - {n_grupos} grupo(s) con {n_reps} apariciones cada uno")

    # Columnas con valores diferentes dentro de al menos un grupo
    cols_auxiliares = {c for c in df_tmp.columns if c.startswith('_llave_') or c.endswith(('_limpio', '_std'))}
    cols_no_llave = [c for c in df.columns if c not in set(llave) | cols_auxiliares]

    cols_con_diferencias = []
    for col in cols_no_llave:
        if df_dup.groupby(llave_norm, dropna=False)[col].nunique(dropna=False).max() > 1:
            cols_con_diferencias.append(col)

    cols_mostrar = ['fila_csv'] + list(llave) + cols_con_diferencias

    if cols_con_diferencias:
        print(f"   Columnas con diferencias entre duplicados: {cols_con_diferencias}")
    else:
        print("   Todos los duplicados son filas 100% idénticas.")

    display(df_dup.sort_values(by=llave_norm)[cols_mostrar].head(20))

# 1. Económico
analizar_duplicados(df_economico, ['NOMBRE_DEL_PACIENTE', 'FECHA_INGRESO_INER'], '1. Económico')

# 2. Comorbilidad
print()
analizar_duplicados(df_comorbilidad, ['nombre', 'fechaing'], '2. Comorbilidad')

# 3. Trabajo Social
print()
df_ts_temp = df_trabajo_social.copy()
for col in ['APELLIDO PATERNO', 'APELLIDO MATERNO', 'NOMBRE']:
    df_ts_temp[col] = df_ts_temp[col].fillna('').str.strip().str.upper()
df_ts_temp['NOMBRE_COMPLETO'] = (
    df_ts_temp['APELLIDO PATERNO'] + " " +
    df_ts_temp['APELLIDO MATERNO'] + " " +
    df_ts_temp['NOMBRE']
)
analizar_duplicados(df_ts_temp, ['NOMBRE_COMPLETO', 'FECHA DE ELABORACIÓN'], '3. Trabajo Social')

___ ANÁLISIS DE DUPLICADOS POR PACIENTE ___

1. Económico: 20 registros duplicados por llave (NOMBRE_DEL_PACIENTE + FECHA_INGRESO_INER)
   Distribución de repeticiones:
   - 10 grupo(s) con 2 apariciones cada uno
   Columnas con diferencias entre duplicados: ['EDAD', 'GRUPO_EDAD', 'RESULTADO', 'ETIQUETAS_COVID', 'FECHA_DE_ALTA_MEJORIA', 'DIAS_ESTANCIA', 'GASTO_TOTAL', 'GASTO_DIARIO', 'TOTAL_DE_INGRESOS', 'TOTAL_DE_EGRESOS', 'ESCOLARIDAD', 'OCUPACION', 'DERECHOHABIENTE_Y/O_BENEFICIARIO', 'VULNERABILIDAD_SOCIOECONOMICA', 'NIVEL_SOCIOECONOMICO']


,fila_csv,NOMBRE_DEL_PACIENTE,FECHA_INGRESO_INER,EDAD,GRUPO_EDAD,RESULTADO,ETIQUETAS_COVID,FECHA_DE_ALTA_MEJORIA,DIAS_ESTANCIA,GASTO_TOTAL,GASTO_DIARIO,TOTAL_DE_INGRESOS,TOTAL_DE_EGRESOS,ESCOLARIDAD,OCUPACION,DERECHOHABIENTE_Y/O_BENEFICIARIO,VULNERABILIDAD_SOCIOECONOMICA,NIVEL_SOCIOECONOMICO
385,387,ALVAREZ RICO GABRIEL,2022-02-11,36.0,35-39,SARS-COV2,COVID,2022-02-13,2,3.265119e+04,16325.595000,NaN,NaN,NaN,NaN,IMSS,False,2
456,458,ALVAREZ RICO GABRIEL,2022-02-11,36.0,35-39,SARS-COV2,COVID,2022-02-13,2,1.525754e+05,76287.700000,NaN,NaN,NaN,NaN,IMSS,False,2
2726,2728,ARGUELLO DE ROSINI ANITA ESTER,2021-10-16,71.0,70-74,SARS-COV2,COVID,2021-10-28,12,2.169483e+05,18079.027333,9500.0,9000.0,SECUNDARIA,NaN,NINGUNO,False,2E
2777,2779,ARGUELLO DE ROSINI ANITA ESTER,2021-10-16,71.0,70-74,SARS-COV2,COVID,2021-10-28,12,2.169483e+05,18079.027333,NaN,NaN,NaN,NaN,NINGUNO,False,NaN
4202,4204,CLEMENTE MILLAN MARIA ALEJANDRA,2020-11-08,49.0,45-49,NaN,NaN,2020-11-21,13,1.568324e+05,12064.034146,7200.0,5912.0,SECUNDARIA,HOGAR,IMSS,True,3
4469,4471,CLEMENTE MILLAN MARIA ALEJANDRA,2020-11-08,49.0,45-49,NaN,NaN,2020-12-04,26,2.216239e+05,8523.995919,7200.0,5912.0,SECUNDARIA,HOGAR,IMSS,True,3
1964,1966,FLORES GUTIERREZ DIEGO GONZALO,2021-06-16,36.0,35-39,SARS-COV2,NaN,2021-06-18,2,1.295955e+05,64797.733850,26500.0,21900.0,NMS,NaN,NINGUNO,True,3
2103,2105,FLORES GUTIERREZ DIEGO GONZALO,2021-06-16,36.0,35-39,SARS-COV2,NaN,2021-07-13,27,5.635057e+05,20870.579915,26500.0,21900.0,NMS,NaN,NINGUNO,True,3
3121,3123,GARCIA VAZQUEZ PATRICIA,2020-05-09,46.0,45-49,SARS-COV2,NaN,2020-05-15,6,1.742756e+05,29045.938750,12100.0,8140.0,SECUNDARIA,HOGAR,NINGUNO,True,2
3372,3374,GARCIA VAZQUEZ PATRICIA,2020-05-09,46.0,45-49,SARS-COV2,NaN,2020-06-30,52,6.972763e+05,13409.158894,12100.0,8140.0,SECUNDARIA,HOGAR,NINGUNO,True,2



2. Comorbilidad: 6 registros duplicados por llave (nombre + fechaing)
   Distribución de repeticiones:
   - 3 grupo(s) con 2 apariciones cada uno
   Columnas con diferencias entre duplicados: ['fechaegr', 'diagnosticoprincipal', 'diagnostico2', 'cie102', 'diagnostico3', 'cie103', 'diagnostico4', 'cie104', 'dx2', 'dx3', 'dx4', 'cardiopatia', 'comorbi', 'diabetes', 'comorbicv']


,fila_csv,nombre,fechaing,fechaegr,diagnosticoprincipal,diagnostico2,cie102,diagnostico3,cie103,diagnostico4,cie104,dx2,dx3,dx4,cardiopatia,comorbi,diabetes,comorbicv
1543,1545,CARLOS PONCE CHAVEZ,2020-12-30,2021-01-05,NEUMONIA POR SARS COV2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Ninguna,0.0,0.0
1887,1889,CARLOS PONCE CHAVEZ,2020-12-30,2021-02-23,SARS-COV2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Ninguna,0.0,0.0
2068,2070,JUAN VICTOR NU?EZ JUAREZ,2021-03-14,2021-03-17,POST COVID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Ninguna,0.0,0.0
2180,2182,JUAN VICTOR NU?EZ JUAREZ,2021-03-14,2021-04-01,NEUMONIA POR SARS COV2,INSUFICIENCIA RESPIRATORIA TIPO I,J96.0,HAS,I10.X,DIABETES MELLITUS TIPO II,E11.9,J96.0,I10.X,E11.9,1.0,Diabetes Mellitus,1.0,HTAS
4113,4115,VICENTE MARTIN VALENCIA CHAVEZ,2023-03-04,2023-03-04,COVID 19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Ninguna,0.0,0.0
4115,4117,VICENTE MARTIN VALENCIA CHAVEZ,2023-03-04,2023-03-06,COVID 19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Ninguna,0.0,0.0



3. Trabajo Social: 4 registros duplicados por llave (NOMBRE_COMPLETO + FECHA DE ELABORACIÓN)
   Distribución de repeticiones:
   - 2 grupo(s) con 2 apariciones cada uno
   Columnas con diferencias entre duplicados: ['FILA', 'EXPEDIENTE', 'EDAD']


,fila_csv,NOMBRE_COMPLETO,FECHA DE ELABORACIÓN,FILA,EXPEDIENTE,EDAD
323,325,GONZALEZ GARCIA JOSE LUIS,18/01/2020 02:44,323,236544,48 años 5 meses 2 días
324,326,GONZALEZ GARCIA JOSE LUIS,18/01/2020 02:44,324,236544,48 años 5 meses 2 días
1350,1352,PRADO APOLINAR ARMANDO,19/04/2020 18:37,1350,237574,46 años 1 meses 16 días
1546,1548,PRADO APOLINAR ARMANDO,19/04/2020 18:37,1546,237771,46 años 2 meses 21 días


### 4.2 Análisis de Registros Vinculados en las 3 bases de datos

In [ ]:
# 1. Función de normalización de nombres
def normalizar_nombre(texto):
    if pd.isna(texto): return ""
    import unicodedata
    s = str(texto).upper().strip()
    s = ''.join((c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn'))
    return " ".join(s.split())

# 2. Estandarización de nombres
df_economico['nombre_std'] = df_economico['NOMBRE_DEL_PACIENTE'].apply(normalizar_nombre)
df_comorbilidad['nombre_std'] = df_comorbilidad['nombre'].apply(normalizar_nombre)
df_trabajo_social['nombre_std'] = (
    df_trabajo_social['APELLIDO PATERNO'].fillna('') + " " +
    df_trabajo_social['APELLIDO MATERNO'].fillna('') + " " +
    df_trabajo_social['NOMBRE'].fillna('')
).apply(normalizar_nombre)

# 3. Crear pares (expediente, nombre_std) por cada base
# Nota: EXP contiene valores no numéricos como 'S/E', los filtramos con pd.to_numeric
df_econ_valid = df_economico.copy()
df_econ_valid['EXP_num'] = pd.to_numeric(df_econ_valid['EXP'], errors='coerce')
df_econ_valid = df_econ_valid.dropna(subset=['EXP_num'])

pares_econ = set(zip(df_econ_valid['EXP_num'].astype(int), df_econ_valid['nombre_std']))
pares_comor = set(zip(
    df_comorbilidad.dropna(subset=['expediente'])['expediente'].astype(int),
    df_comorbilidad.dropna(subset=['expediente'])['nombre_std']
))
pares_ts = set(zip(
    df_trabajo_social.dropna(subset=['EXPEDIENTE'])['EXPEDIENTE'].astype(int),
    df_trabajo_social.dropna(subset=['EXPEDIENTE'])['nombre_std']
))

# 4. Extraer conjuntos individuales de expedientes y nombres
exp_econ = {p[0] for p in pares_econ}
exp_comor = {p[0] for p in pares_comor}
exp_ts = {p[0] for p in pares_ts}

nom_econ = {p[1] for p in pares_econ}
nom_comor = {p[1] for p in pares_comor}
nom_ts = {p[1] for p in pares_ts}

# 5. Análisis por pares (con métrica corregida)
comparaciones = [
    ('Económico', 'Comorbilidad', exp_econ, exp_comor, nom_econ, nom_comor, pares_econ, pares_comor),
    ('Económico', 'Trabajo Social', exp_econ, exp_ts, nom_econ, nom_ts, pares_econ, pares_ts),
    ('Comorbilidad', 'Trabajo Social', exp_comor, exp_ts, nom_comor, nom_ts, pares_comor, pares_ts),
]

resumen_vinculacion = []

for nombre_a, nombre_b, exp_a, exp_b, nom_a, nom_b, par_a, par_b in comparaciones:
    coinciden_exp = exp_a & exp_b
    coinciden_nom = nom_a & nom_b
    coinciden_par = par_a & par_b
    # Métrica corregida: contar expedientes únicos con al menos un par exacto
    exp_con_match = {p[0] for p in coinciden_par}
    exp_sin_match = len(coinciden_exp) - len(exp_con_match)

    resumen_vinculacion.append({
        'Par': f'{nombre_a} <-> {nombre_b}',
        'Únicos A': len(exp_a),
        'Únicos B': len(exp_b),
        'Coinciden por Expediente': len(coinciden_exp),
        'Coinciden por Nombre': len(coinciden_nom),
        'Coinciden por Ambos (Exp+Nombre)': len(exp_con_match),
        'Exp. compartido, Nombre diferente': exp_sin_match,
    })

df_vinculacion = pd.DataFrame(resumen_vinculacion)
print("___ ANÁLISIS DE VINCULACIÓN POR PARES (EXPEDIENTE + NOMBRE) ___")
display(df_vinculacion)

# 6. Detalle de inconsistencias: mismo expediente pero nombre diferente
for nombre_a, nombre_b, exp_a, exp_b, nom_a, nom_b, par_a, par_b in comparaciones:
    exp_compartidos = exp_a & exp_b
    pares_coincidentes = par_a & par_b
    exp_con_match = {p[0] for p in pares_coincidentes}
    exp_sin_match_nombre = exp_compartidos - exp_con_match

    if exp_sin_match_nombre:
        print(f"\n[!] {nombre_a} <-> {nombre_b}: {len(exp_sin_match_nombre)} expedientes compartidos con nombre DIFERENTE")
        muestra = list(exp_sin_match_nombre)[:10]
        nombres_a = {p[0]: p[1] for p in par_a if p[0] in muestra}
        nombres_b = {p[0]: p[1] for p in par_b if p[0] in muestra}
        discrepancias = pd.DataFrame([
            {'Expediente': e, f'Nombre en {nombre_a}': nombres_a.get(e, ''), f'Nombre en {nombre_b}': nombres_b.get(e, '')}
            for e in muestra
        ])
        display(discrepancias)
    else:
        print(f"\n[OK] {nombre_a} <-> {nombre_b}: Todos los expedientes compartidos tienen el mismo nombre normalizado.")


___ ANÁLISIS DE VINCULACIÓN PAIRWISE (EXPEDIENTE + NOMBRE) ___


,Par,Únicos A,Únicos B,Coinciden por Expediente,Coinciden por Nombre,Coinciden por Ambos (Exp+Nombre),"Exp. compartido, Nombre diferente"
0,Económico <-> Comorbilidad,4330,4097,3629,1,1,3628
1,Económico <-> Trabajo Social,4330,14795,3995,3661,3653,342
2,Comorbilidad <-> Trabajo Social,4097,14795,3800,17,16,3784



[!] Económico <-> Comorbilidad: 3628 expedientes compartidos con nombre DIFERENTE


,Expediente,Nombre en Económico,Nombre en Comorbilidad
0,237568,MOLINA GONZALEZ BLANCA ESTELA,BLANCA ESTELA MOLINA GONZALEZ
1,237569,MENDEZ AGUILAR JOSE ODILON MAR,JOSE ODILON MAR MENDEZ AGUILAR
2,237571,GARCIA MARTINEZ DONACIANO,DONACIANO GARCIA MARTINEZ
3,237572,CALDERON REYES CONSUELO,CONSUELO CALDERON REYES
4,237574,PRADO APOLINAR VICTOR DANIEL,DANIEL PRADO APOLINAR
5,237577,ROSAS CHAVARRIA GUILLERMO,GUILLERMO ROSAS CHAVARRIA
6,237578,VIDAL SEGUNDO ESTEBAN,ESTEBAN VIDAL SEGUNDO
7,237579,GARDUNO TREJO ALEJANDRO,ALEJANDRO GARDU?O TREJO
8,237581,GUADARRAMA PINA MARIA DEL CARME,MARIA DEL CARME GUADARRAMA PI?A
9,237582,CASADO CORONA JUAN JOSE,JUAN JOSE CASADO CORONA



[!] Económico <-> Trabajo Social: 342 expedientes compartidos con nombre DIFERENTE


,Expediente,Nombre en Económico,Nombre en Trabajo Social
0,239616,PRIETO CENADO BERNANDO,PRIETO CENADO BERNARDO
1,237569,MENDEZ AGUILAR JOSE ODILON MAR,MENDEZ AGUILAR JOSE ODILON MARIO
2,237574,PRADO APOLINAR VICTOR DANIEL,PRADO APOLINAR ARMANDO
3,239623,MARTINEZ GARCIA CONCEPCION,MARTINEZ GARCIA MARIA CONCEPCION
4,237581,GUADARRAMA PINA MARIA DEL CARME,GUADARRAMA PINA MARIA DEL CARMEN
5,243739,LOPEZ NOLASCO RENATA ALEJANDRA,RENATA ALEJANDRA LOPEZ NOLASCO
6,239654,STEINNER REYES JUAN FEDERICO,STENNER REYES JUAN FEDERICO
7,237610,MARTINEZ LOPEZ CRISTELA JAZMIN,MARTINEZ LOPEZ CRISTELA YAZMIN
8,245802,ESPINOSA CORTES LUIS,ESPINOSA CORTEZ LUIS
9,239659,CONTRERAS VALDEZ MARIA GUADALUPE,CONTRERAS VALDES MARIA GUADALUPE



[!] Comorbilidad <-> Trabajo Social: 3784 expedientes compartidos con nombre DIFERENTE


,Expediente,Nombre en Comorbilidad,Nombre en Trabajo Social
0,237568,BLANCA ESTELA MOLINA GONZALEZ,MOLINA GONZALEZ BLANCA ESTELA
1,237569,JOSE ODILON MAR MENDEZ AGUILAR,MENDEZ AGUILAR JOSE ODILON MARIO
2,237571,DONACIANO GARCIA MARTINEZ,GARCIA MARTINEZ DONACIANO
3,237572,CONSUELO CALDERON REYES,CALDERON REYES CONSUELO
4,237573,FRANCISCO CASTRO SEGURA,CASTRO SEGURA FRANCISCO
5,237574,DANIEL PRADO APOLINAR,PRADO APOLINAR ARMANDO
6,237575,LETICIA RODRIGUEZ LUGO,RODRIGUEZ LUGO LETICIA
7,237576,FERMIN VARGAS PEREZ,VARGAS PEREZ FERMIN
8,237577,GUILLERMO ROSAS CHAVARRIA,ROSAS CHAVARRIA GUILLERMO
9,237578,ESTEBAN VIDAL SEGUNDO,VIDAL SEGUNDO ESTEBAN


### 4.3 Análisis de caracteres problemáticos en campos de nombre (todas las bases)

In [20]:
# Diagnóstico de caracteres problemáticos en campos de nombre (todas las bases)
import re

def detectar_caracteres_problematicos(serie, nombre_base):
    """Detecta caracteres no alfabéticos/espacios en una serie de nombres."""
    patron = re.compile(r'[^A-ZÁÉÍÓÚÜÑ \-]', re.IGNORECASE)
    problemas = serie.dropna().apply(lambda x: set(patron.findall(str(x))))
    problemas = problemas[problemas.apply(len) > 0]
    if problemas.empty:
        return pd.DataFrame()
    # Contar frecuencia de cada carácter problemático
    from collections import Counter
    conteo = Counter()
    for chars in problemas:
        conteo.update(chars)
    df_resultado = pd.DataFrame([
        {'Base': nombre_base, 'Carácter': c, 'repr': repr(c), 'Ocurrencias en registros': n}
        for c, n in conteo.most_common()
    ])
    return df_resultado

print("\n___ DIAGNÓSTICO DE CARACTERES PROBLEMÁTICOS EN NOMBRES ___")
diagnostico = pd.concat([
    detectar_caracteres_problematicos(df_economico['NOMBRE_DEL_PACIENTE'], 'Económico'),
    detectar_caracteres_problematicos(df_comorbilidad['nombre'], 'Comorbilidad'),
    detectar_caracteres_problematicos(
        df_trabajo_social['APELLIDO PATERNO'].fillna('') + " " +
        df_trabajo_social['APELLIDO MATERNO'].fillna('') + " " +
        df_trabajo_social['NOMBRE'].fillna(''),
        'Trabajo Social'
    ),
], ignore_index=True)

if diagnostico.empty:
    print("[OK] No se detectaron caracteres problemáticos.")
else:
    display(diagnostico)
    # Muestra de registros afectados por '?' (encoding roto)
    for base_nombre, df_base, col_nombre in [
        ('Económico', df_economico, 'NOMBRE_DEL_PACIENTE'),
        ('Comorbilidad', df_comorbilidad, 'nombre'),
    ]:
        afectados = df_base[df_base[col_nombre].str.contains(r'\?', na=False)]
        if len(afectados) > 0:
            print(f"\n[?] {base_nombre}: {len(afectados)} registros con '?' en el nombre:")
            display(afectados[[col_nombre]].head(10))


___ DIAGNÓSTICO DE CARACTERES PROBLEMÁTICOS EN NOMBRES ___


,Base,Carácter,repr,Ocurrencias en registros
0,Económico,.,'.',49
1,Económico,(,'(',28
2,Económico,),')',28
3,Económico,/,'/',1
4,Económico,1,'1',1
5,Económico,6,'6',1
6,Comorbilidad,?,'?',223
7,Comorbilidad,.,'.',85
8,Comorbilidad,|,'|',3
9,Comorbilidad,5,'5',1



[?] Comorbilidad: 223 registros con '?' en el nombre:


,nombre
28,JUAN RAMIREZ NI?O
29,GRISELDA PAEZ NI?O
39,BERTHA NU?EZ REYES
47,REMEDIOS GARDU?O MATA
51,ABIGAIL AQUINO COUTI?O
82,RICARDO EDUARDO CEDE?O URBINA
87,HECTOR BOLA?OS MARTINEZ
94,MARIA DEL CARME GUADARRAMA PI?A
109,GLORIA PE?A MONTES
114,SAMUEL ARTURO GARDU?O ESCOBEDO
